In [ ]:
!pip install botasaurus global_land_mask beautifulsoup4

In [ ]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

In [ ]:
import os
import re
import csv
import time
import random
from botasaurus.browser import browser, Driver
from botasaurus.soupify import soupify
from global_land_mask import globe
from google.colab import drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CITY_NAME = "HaNoi"
PART_TO_RUN = 1
BATCH_SIZE = 10

BASE_DIR = '/content/drive/MyDrive/SmartSite/Step 1: Extract'
CSV_DIR = os.path.join(BASE_DIR, 'Results_CSV')
HISTORY_DIR = os.path.join(BASE_DIR, 'History_TXT')

os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(CSV_DIR, f"{CITY_NAME}_GGMap_Part{PART_TO_RUN}.csv")
HISTORY_FILE = os.path.join(HISTORY_DIR, f"{CITY_NAME}_History_Part{PART_TO_RUN}.txt")

print(f"✅ Đã cấu hình xong cho {CITY_NAME} - Luồng {PART_TO_RUN}")
print(f"✅ Lưu kết quả vào {OUTPUT_FILE}")
print(f"✅ Lưu lịch sử vào {HISTORY_FILE}")

✅ Đã cấu hình xong cho HaNoi - Luồng 1
✅ Lưu kết quả vào /content/drive/MyDrive/SmartSite/Step 1: Extract/Results_CSV/HaNoi_GGMap_Part1.csv
✅ Lưu lịch sử vào /content/drive/MyDrive/SmartSite/Step 1: Extract/History_TXT/HaNoi_History_Part1.txt


In [ ]:
# ==============================================================================
# 1. CẤU HÌNH TỪ KHÓA
# ==============================================================================
SEARCH_KEYWORDS = ["Quán cà phê", "Coffee shop", "Cafe"]
VALID_CATEGORIES = [
    "Coffee shop", "Cafe", "Espresso bar", "Tea house", "Bubble tea store",
    "Quán cà phê", "Quán trà sữa", "Dessert shop", "Bán cà phê"
]
COFFEE_KEYWORDS = [
    "cafe", "coffee", "cà phê", "trà sữa", "milk tea",
    "tea", "trà", "roastery", "cacao", "kem", "dessert"
]
BLACKLIST_KEYWORDS = [
    "nhậu", "bia", "beer", "pub", "bar", "restaurant", "nhà hàng",
    "cơm", "bún", "phở", "mì quảng", "hủ tiếu", "cháo", "homestay",
    "hotel", "spa", "nails", "tạp hóa", "siêu thị", "pharmacy", "massage", "karaoke", "bia hơi"
]


# ==============================================================================
# 2. HÀM QUẢN LÝ LƯỚI TỰ ĐỘNG - CHUYÊN BIỆT CHO HÀ NỘI
# ==============================================================================
def get_bounds_for_part(part):
    mid_lat, mid_lng = 20.9750, 105.6500
    if part == 1: return {"min_lat": mid_lat, "max_lat": 21.4000, "min_lng": 105.2800, "max_lng": mid_lng}
    elif part == 2: return {"min_lat": mid_lat, "max_lat": 21.4000, "min_lng": mid_lng, "max_lng": 106.0200}
    elif part == 3: return {"min_lat": 20.5500, "max_lat": mid_lat, "min_lng": 105.2800, "max_lng": mid_lng}
    elif part == 4: return {"min_lat": 20.5500, "max_lat": mid_lat, "min_lng": mid_lng, "max_lng": 106.0200}
    return {"min_lat": 20.5500, "max_lat": 21.4000, "min_lng": 105.2800, "max_lng": 106.0200}


def generate_adaptive_grid():
    points = []
    bounds = get_bounds_for_part(PART_TO_RUN)

    CORE_BOUNDS = {"min_lat": 20.9800, "max_lat": 21.0800, "min_lng": 105.7700, "max_lng": 105.8800}
    STEP_CORE, STEP_SUBURB = 0.005, 0.015

    lat = bounds["min_lat"]
    while lat <= bounds["max_lat"]:
        lng = bounds["min_lng"]
        while lng <= bounds["max_lng"]:
            if globe.is_land(lat, lng):
                in_core = (CORE_BOUNDS["min_lat"] <= lat <= CORE_BOUNDS["max_lat"]) and (
                        CORE_BOUNDS["min_lng"] <= lng <= CORE_BOUNDS["max_lng"])
                if not in_core: points.append((lat, lng))
            lng += STEP_SUBURB
        lat += STEP_SUBURB

    lat_core = max(bounds["min_lat"], CORE_BOUNDS["min_lat"])
    max_lat_core = min(bounds["max_lat"], CORE_BOUNDS["max_lat"])
    while lat_core <= max_lat_core:
        lng_core = max(bounds["min_lng"], CORE_BOUNDS["min_lng"])
        max_lng_core = min(bounds["max_lng"], CORE_BOUNDS["max_lng"])
        while lng_core <= max_lng_core:
            if globe.is_land(lat_core, lng_core): points.append((lat_core, lng_core))
            lng_core += STEP_CORE
        lat_core += STEP_CORE

    unique_points = list(set(points))
    random.shuffle(unique_points)
    return unique_points

    # ==============================================================================
# 3. QUẢN LÝ FILE
# ==============================================================================
def load_existing_ids():
    ids = set()
    try:
        if os.path.exists(OUTPUT_FILE):
            with open(OUTPUT_FILE, 'r', encoding='utf-8-sig') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if 'place_id' in row: ids.add(row['place_id'])
            print(f"🔄 Đã load {len(ids)} quán cũ từ CSV.")
    except Exception as e:
        print(f"Lỗi đọc file CSV: {e}")
    return ids


def load_history():
    points = set()
    try:
        if os.path.exists(HISTORY_FILE):
            with open(HISTORY_FILE, 'r', encoding='utf-8') as f:
                for line in f: points.add(line.strip())
            print(f"🔄 Đã load {len(points)} điểm đã quét.")
    except Exception as e:
        print(f"Lỗi đọc file History: {e}")
    return points


def append_history(lat, lng):
    try:
        with open(HISTORY_FILE, 'a', encoding='utf-8') as f:
            f.write(f"{lat}_{lng}\n")
    except Exception as e:
        print(f"⚠️ Lỗi ghi file txt: {e}")


def save_batch(batch_data):
    if not batch_data: return
    try:
        file_exists = os.path.isfile(OUTPUT_FILE)
        with open(OUTPUT_FILE, mode='a', newline='', encoding='utf-8-sig') as f:
            fieldnames = [
                "name", "address", "phone", "category", "rating",
                "reviews_count", "status", "opening_hours", "sample_reviews",
                "lat", "lng", "grid_lat", "grid_lng", "google_url", "place_id"
            ]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            if not file_exists:
                writer.writeheader()
            writer.writerows(batch_data)
        print(f"    💾 [SAVED] Đã lưu {len(batch_data)} quán vào Drive.")
    except Exception as e:
        print(f"⚠️ Lỗi ghi file csv: {e}")

# ==============================================================================
# 4. BỘ LỌC VÀ LẤY DỮ LIỆU
# ==============================================================================
def is_valid_cafe(name, category):
    name_lower, cat_lower = name.lower(), category.lower()
    for bad in BLACKLIST_KEYWORDS:
        if bad in name_lower or bad in cat_lower: return False
    for cat in VALID_CATEGORIES:
        if cat.lower() in cat_lower: return True
    if any(g in cat_lower for g in ["store", "cửa hàng", "shop"]):
        if any(kw in name_lower for kw in COFFEE_KEYWORDS): return True
    return False


def get_place_id(url):
    try:
        return url.split("!1s")[1].split("!")[0] if "!1s" in url else url
    except:
        return url


def extract_real_lat_lng(url):
    try:
        match = re.search(r'!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)', url)
        if match:
            return float(match.group(1)), float(match.group(2))
        match = re.search(r'/@(-?\d+\.\d+),(-?\d+\.\d+)', url)
        if match:
            return float(match.group(1)), float(match.group(2))
    except:
        pass
    return None, None


def extract_hours(driver, soup):
    try:
        hours_btn = soup.select_one('[data-item-id="oh"] div[role="button"]')
        if hours_btn and hours_btn.get('aria-label'): return hours_btn.get('aria-label').replace(
            "Hide open hours for the week", "").strip()
        status_div = soup.select_one('[data-item-id="oh"]')
        if status_div: return status_div.get_text(strip=True)
    except:
        pass
    return "N/A"


def extract_random_reviews(driver):
    try:
        # 1. TÌM VÀ CLICK TAB ĐÁNH GIÁ
        review_btns = driver.select_all('button[role="tab"]')
        target_btn = next((btn for btn in review_btns if btn.text and (
            "review" in btn.text.lower() or "đánh giá" in btn.text.lower())), None)

        if not target_btn:
            return ""

        target_btn.click()
        time.sleep(random.uniform(2.5, 3.5))

        # 2. CUỘN TRANG (Tải thêm review)
        try:
            for _ in range(4):
                driver.press_key('PageDown')
                time.sleep(random.uniform(1.2, 1.8))
        except:
            pass

        # 3. CHỐNG CẮT CỤT: TÌM VÀ BẤM TẤT CẢ CÁC NÚT "THÊM" (MORE)
        try:
            # Nút "Thêm" trên GG Maps thường dùng class w8nwRe
            more_btns = driver.select_all('button.w8nwRe')
            for btn in more_btns:
                try:
                    btn.click()
                    time.sleep(0.1) # Bấm nhanh qua các nút
                except:
                    pass
            time.sleep(1) # Đợi 1 chút cho text bung ra hết
        except:
            pass

        # 4. TRÍCH XUẤT DỮ LIỆU TỪ TỪNG KHỐI REVIEW
        soup = soupify(driver)
        temp_reviews = []

        # 'div.jftiEf' là class bao trọn 1 cái review (gồm cả Avatar, Tên, Sao, Text)
        review_blocks = soup.select('div.jftiEf')

        for block in review_blocks:
            try:
                # 4.1 Lấy Text
                content = block.select_one('.wiI7pd')
                if not content: continue
                text = content.get_text(strip=True)

                # BỘ LỌC CHẤT LƯỢNG
                if len(text) > 15 and text not in [r.split('] ')[-1] for r in temp_reviews]:

                    # 4.2 Lấy Số Sao (Chắc chắn 100% bắt được)
                    stars = "?"
                    # Thẻ chứa sao thường có role="img" hoặc aria-label chứa từ khóa
                    rating_el = block.select_one('span[aria-label*="star"], span[aria-label*="sao"], span[role="img"]')

                    if rating_el and rating_el.has_attr('aria-label'):
                        # Dùng regex bóc đúng con số ra (VD: "5 sao" -> "5")
                        match = re.search(r'(\d+)', rating_el['aria-label'])
                        if match:
                            stars = match.group(1)

                    temp_reviews.append(f"[{stars} sao] {text}")
            except:
                continue

        if temp_reviews:
            limit = min(len(temp_reviews), 15)
            return " || ".join(temp_reviews[:limit])

    except Exception as e:
        pass

    return ""


def parse_place_full(driver, url, lat_scan, lng_scan):
    try:
        driver.get(url)
        # Tối ưu thời gian chờ load quán (Vừa đủ nhanh, vừa đủ an toàn)
        time.sleep(random.uniform(1.8, 2.5))
        soup = soupify(driver)

        try:
            name = soup.select_one('h1').get_text(strip=True)
        except:
            return None
        try:
            category = soup.select_one('button[jsaction*="category"]').get_text(strip=True)
        except:
            category = "Store"

        if not is_valid_cafe(name, category): return None

        try:
            address = soup.select_one('[data-item-id="address"]').get_text(strip=True)
        except:
            address = ""
        try:
            phone = soup.select_one('[data-item-id*="phone"]').get_text(strip=True)
        except:
            phone = ""
        try:
            rating = soup.select_one('div.F7nice span[aria-hidden="true"]').get_text(strip=True)
        except:
            rating = ""

        # =========================================================
        # BỘ LỌC SỐ LƯỢNG ĐÁNH GIÁ (REVIEW COUNT) TOÀN DIỆN
        # =========================================================
        try:
            reviews_count = "0"

            # Cách 1: Tìm trong khối Rating chính (F7nice)
            rating_block = soup.select_one('div.F7nice')
            if rating_block:
                full_text = rating_block.get_text(separator=' ', strip=True).lower()

                # Regex tìm 2 trường hợp:
                # 1. Nằm trong ngoặc: (34) hoặc (1,234)
                # 2. Đứng trước chữ: 34 reviews, 34 đánh giá, 1.234 bài đánh giá
                match = re.search(r'\(([\d\.,]+)\)|([\d\.,]+)\s*(?:review|đánh giá)', full_text)
                if match:
                    raw_num = match.group(1) if match.group(1) else match.group(2)
                    if raw_num:
                        reviews_count = re.sub(r'[^\d]', '', raw_num)

            # Cách 2: Quét TẤT CẢ các nút (button) trên trang
            if not reviews_count or reviews_count == "0":
                for btn in soup.select('button'):
                    btn_text = btn.get_text(strip=True).lower()
                    if 'review' in btn_text or 'đánh giá' in btn_text:
                        nums = re.findall(r'[\d\.,]+', btn_text)
                        if nums:
                            reviews_count = re.sub(r'[^\d]', '', nums[0])
                            if reviews_count: break

            # Cách 3: Quét tự do trên toàn bộ text của trang
            if not reviews_count or reviews_count == "0":
                body_text = soup.get_text(separator=' ').lower()
                match_body = re.search(r'([\d\.,]+)\s*(?:reviews|đánh giá|bài đánh giá)', body_text)
                if match_body:
                    reviews_count = re.sub(r'[^\d]', '', match_body.group(1))

            if not reviews_count:
                reviews_count = "0"

        except Exception as e:
            reviews_count = "0"
        # =========================================================

        status = "Closed" if "Permanently closed" in soup.get_text() or "Đã đóng cửa vĩnh viễn" in soup.get_text() else "Open"
        hours = extract_hours(driver, soup)
        sample_reviews = extract_random_reviews(driver) if status == "Open" else ""

        real_lat, real_lng = extract_real_lat_lng(url)

        return {
            "name": name,
            "address": address,
            "phone": phone,
            "category": category,
            "rating": rating,
            "reviews_count": reviews_count,
            "status": status,
            "opening_hours": hours,
            "sample_reviews": sample_reviews,

            "lat": real_lat if real_lat else lat_scan,
            "lng": real_lng if real_lng else lng_scan,

            "grid_lat": lat_scan,
            "grid_lng": lng_scan,
            "google_url": url,
            "place_id": get_place_id(url)
        }
    except:
        return None

In [ ]:
import json

@browser(block_images=True, reuse_driver=True, headless=True)
def run_quick_test(driver: Driver, data):
    print("🛠️ BẮT ĐẦU TEST CRAWL (Chỉ lấy 2 quán để kiểm tra)...")

    # Lấy thử tọa độ đầu tiên của luồng này
    test_grid = generate_adaptive_grid()
    if not test_grid:
        print("Lưới rỗng!")
        return

    lat, lng = test_grid[0]
    print(f"\n📍 Đang test tại Grid: {lat:.4f}, {lng:.4f}")

    # Chỉ dùng 1 từ khóa đầu tiên để test cho lẹ
    kw = SEARCH_KEYWORDS[0]
    driver.get(f"https://www.google.com/maps/search/{kw}/@{lat},{lng},15z")
    time.sleep(3)

    try:
        driver.scroll('div[role="feed"]')
        time.sleep(1)
    except: pass

    elements = driver.select_all('a[href*="/maps/place/"]')

    test_results = []
    for el in elements:
        url = el.get_attribute('href')
        if not url: continue

        print(f" ⏳ Đang bóc tách 1 quán...")
        place_data = parse_place_full(driver, url, lat, lng)

        if place_data:
            test_results.append(place_data)
            print(f" ✅ THÀNH CÔNG: {place_data['name']}")

        if len(test_results) >= 2: # Lấy đúng 2 cái rồi dừng
            break

    print("\n" + "="*50)
    print("📊 KẾT QUẢ DỮ LIỆU CỦA 2 QUÁN (CHECK ĐỦ 15 CỘT CHƯA NHA):")
    print("="*50)
    for res in test_results:
        print(json.dumps(res, indent=4, ensure_ascii=False))

# Chạy Test
run_quick_test()

🛠️ BẮT ĐẦU TEST CRAWL (Chỉ lấy 2 quán để kiểm tra)...

📍 Đang test tại Grid: 20.9750, 105.2950
 ⏳ Đang bóc tách 1 quán...
 ✅ THÀNH CÔNG: Trà sữa chang phi
 ⏳ Đang bóc tách 1 quán...
 ✅ THÀNH CÔNG: MilkTea 1991

📊 KẾT QUẢ DỮ LIỆU CỦA 2 QUÁN (CHECK ĐỦ 15 CỘT CHƯA NHA):
{
    "name": "Trà sữa chang phi",
    "address": "cầu trắng yên sơn thanh sơn, Phú thọ, Phú Thọ 35800, Vietnam",
    "phone": "+84 334 790 796",
    "category": "Coffee shop",
    "rating": "5.0",
    "reviews_count": "3",
    "status": "Open",
    "opening_hours": "N/A",
    "sample_reviews": "[5 sao] Chang Phi Milk Tea has a decent, clean, and airy space. The drink menu is diverse with many attractive milk tea flavors at reasonable prices. The staff are quick and efficient. To add more excitement for customers, Chang Phi Milk Tea could consider using Tuneast. Customers could freely choose their favorite songs, creating a fun and lively musical atmosphere tailored to their preferences.",
    "lat": 20.9723644,
    "lng

In [ ]:
@browser(block_images=True, reuse_driver=True, close_on_crash=True, headless=True, output=None)
def run_final_crawler(driver: Driver, data):
    print(f"🚀 BẮT ĐẦU CRAWL LUỒNG {PART_TO_RUN} - HÀ NỘI...")

    crawled_ids = load_existing_ids()
    scanned_history = load_history()
    all_grid_points = generate_adaptive_grid()

    grid_points = [(lat, lng) for lat, lng in all_grid_points if f"{lat}_{lng}" not in scanned_history]
    print(f"📊 Tổng Grid luồng {PART_TO_RUN}: {len(all_grid_points)}. Cần chạy: {len(grid_points)}")

    current_batch = []

    for i, (lat, lng) in enumerate(grid_points):
        print(f"\n📍 Grid [{i + 1}/{len(grid_points)}]: {lat:.4f}, {lng:.4f}")
        found_count = 0

        for kw in SEARCH_KEYWORDS:
            try:
                driver.get(f"https://www.google.com/maps/search/{kw}/@{lat},{lng},14z")
                # Tối ưu thời gian chờ kết quả tìm kiếm
                time.sleep(random.uniform(2.5, 3.5))

                try:
                    if driver.select('div[role="feed"]'):
                        for _ in range(3):
                            driver.scroll('div[role="feed"]')
                            time.sleep(random.uniform(0.8, 1.2))
                except:
                    pass

                elements = driver.select_all('a[href*="/maps/place/"]')

                for el in elements:
                    url = el.get_attribute('href')
                    if not url: continue
                    p_id = get_place_id(url)

                    if p_id in crawled_ids or any(item['place_id'] == p_id for item in current_batch): continue

                    data = parse_place_full(driver, url, lat, lng)
                    if data:
                        current_batch.append(data)
                        crawled_ids.add(p_id)
                        found_count += 1
                        print(f" ✅ LẤY: {data['name']} | ⭐ {data['rating']} | 📝 {data['reviews_count']} đánh giá")
                    else:
                        crawled_ids.add(p_id)

                    if len(current_batch) >= BATCH_SIZE:
                        save_batch(current_batch)
                        current_batch.clear()

            except Exception as e:
                pass

        append_history(lat, lng)
        if found_count == 0: print("   ⚠️ Không có quán mới.")

    if current_batch: save_batch(current_batch)
    print("\n🎉 HOÀN TẤT VÙNG NÀY!")


if __name__ == "__main__":
    run_final_crawler()